# `apptbl-hyperparameters`: choice-training hyperparameters shared by every trained run

In [1]:
from pathlib import Path
import json
import sys

from IPython.display import display
import pandas as pd

ASSET_NAME = "apptbl-hyperparameters"
ASSET_DIR = Path.cwd()
PAPER_DIR = ASSET_DIR.parent
SRC_DIR = PAPER_DIR / "src"
if not (SRC_DIR / "microdata_analysis.py").is_file():
    raise FileNotFoundError(f"Open this notebook from assets/; {SRC_DIR} has no microdata_analysis.py")
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import microdata_analysis as ma

print(f"Outputs go to {ASSET_DIR / ASSET_NAME}.*")

Outputs go to /home/dw/github/decision-models-finetuning/code/train-notebooks/beta-training/ms_paper_beta_loss/manuscript-training-assets/apptbl-hyperparameters.*


## Run design

In [2]:
RUN_ROOTS = (PAPER_DIR / "runs-rorqual", PAPER_DIR / "runs-nibi")
MODELS = ("kev-0.8b", "kev-4b", "kev-9b")
TRAINED_SLUGS = ("row2b_wu_ce_oracle", "row3_berke_ce_only", "row5_full_berke_wu_ce_oracle")
SELECTION_VALIDATOR = "kriner2026"

# run.json of every trained run, keyed by (model, regime)
run_records = {}
for model_slug in MODELS:
    for regime_slug in TRAINED_SLUGS:
        run_dir = ma.find_run_dir(RUN_ROOTS, model_slug, regime_slug)
        run_record = json.loads((run_dir / "run.json").read_text(encoding="utf-8"))
        if run_record["status"] != "finished":
            raise ValueError(f"{run_dir} status is {run_record['status']!r}, expected 'finished'")
        run_records[(model_slug, regime_slug)] = run_record
print(f"Loaded {len(run_records)} finished runs")

Loaded 9 finished runs


## Settings fixed in code rather than stored in run.json

In [3]:
ma.setup_analysis_env()
from beta_train import trainer as beta_trainer

# kev/kev/model.py builds the LoRA adapters with lora_alpha = 2 * rank and lora_dropout = 0.05.
LORA_ALPHA_PER_RANK = 2
LORA_DROPOUT = 0.05
# The trainer passes no betas or eps to torch.optim.AdamW, so PyTorch's defaults apply.
ADAMW_BETAS = (0.9, 0.999)
ADAMW_EPS = 1e-8
# torch.optim.lr_scheduler.OneCycleLR default anneal_strategy
ONECYCLE_ANNEAL = "cosine"
# slurm/02-train-paper-9b.sbatch requests --gpus=h100:1
HARDWARE_TEXT = "One NVIDIA H100 (80 GB) per run"
# Pointer-head projection weights in trainable_shapes
HEAD_WEIGHT_KEY = "net.head.q.weight"

ONECYCLE_PCT_START = beta_trainer.ONECYCLE_PCT_START
ONECYCLE_DIV_FACTOR = beta_trainer.ONECYCLE_DIV_FACTOR
ONECYCLE_FINAL_DIV_FACTOR = beta_trainer.ONECYCLE_FINAL_DIV_FACTOR

## Labels and formatting (edit wording here)

In [4]:
MODEL_LABELS = {"kev-0.8b": "KEV-0.8B", "kev-4b": "KEV-4B", "kev-9b": "KEV-9B"}
REGIME_LABELS = {
    "row2b_wu_ce_oracle": "Electricity",
    "row3_berke_ce_only": "Drones",
    "row5_full_berke_wu_ce_oracle": "Drones + Electricity",
}
VALIDATOR_LABEL = "Flights"
DTYPE_LABELS = {"bf16": "bfloat16"}
LR_SCHEDULE_LABELS = {"onecycle": "OneCycle"}
# LoRA module names in trainable_shapes, shown as one phrase
EXPECTED_LORA_MODULES = {
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
    "in_proj_qkv", "in_proj_z", "in_proj_a", "in_proj_b", "out_proj",
}
LORA_MODULES_TEXT = "All attention, MLP, and Gated DeltaNet projections"

HEADERS = ["Setting", "Value"]
COLUMN_RULES = [":---", ":---"]

## Format as a Pandoc pipe table

In [5]:
import math
import re

from matplotlib import font_manager
from PIL import ImageFont

# Page and table style of manuscript/quarto-files/ms_template.docx. Word fixes the column widths that Pandoc
# derives from the rule dash counts, so a column narrower than its longest word splits that word across lines.
TEXT_WIDTH_PT = 468.0  # 8.5 in page minus 1 in left and right margins
TABLE_FONT_PT = 10.0  # Compact paragraph style used in table cells
CELL_PADDING_PT = 10.8  # Table style left + right cell margins (108 twips each)
COLUMN_SLACK_PT = 2.0  # per-column allowance for differences between Word and the font metrics measured here
MATH_WIDTH_FACTOR = 1.15  # Word sets $...$ in Cambria Math, which runs wider than Times New Roman
# No-break spaces around ×: Word sets each $...$ as one equation that never wraps.
MATH_SYMBOLS = {r"\beta": "β", r"\alpha": "α", r"\epsilon": "ε", r"\lvert": "|", r"\rvert": "|", r"\times": "\u00a0×\u00a0"}


def load_table_font(weight):
    font_properties = font_manager.FontProperties(family="Times New Roman", weight=weight)
    font_path = font_manager.findfont(font_properties, fallback_to_default=False)
    return ImageFont.truetype(font_path, 1000)


REGULAR_FONT = load_table_font("normal")
BOLD_FONT = load_table_font("bold")


def shown_math(match):
    latex = match.group(1)
    text = latex
    for command, symbol in MATH_SYMBOLS.items():
        text = text.replace(command, symbol)
    text = text.replace(r"\mathrm", "")
    for mark in ("{", "}", "_", "^"):
        text = text.replace(mark, "")
    if "\\" in text:
        raise ValueError(f"No display text for a LaTeX command in {latex!r}; add it to MATH_SYMBOLS")
    return text


def shown_citation(match):
    surname = match.group(1).title()
    year = match.group(2)
    return f"({surname} {year})"


def shown_cell(cell):
    """Text Word displays for a markdown cell, with the font and width factor to measure it."""
    is_bold = cell.startswith("**") and cell.endswith("**")
    text = cell
    if is_bold:
        text = text[2:-2]
    text = re.sub(r"\[@([a-z]+)(\d{4})[a-z0-9]*\]", shown_citation, text)
    has_math = re.search(r"(?<!\\)\$", text) is not None
    text = re.sub(r"(?<!\\)\$(.+?)(?<!\\)\$", shown_math, text)
    text = text.replace(r"\$", "$")
    text = text.replace(r"\|", "|")
    font = BOLD_FONT if is_bold else REGULAR_FONT
    factor = MATH_WIDTH_FACTOR if has_math else 1.0
    return text, font, factor


def text_width_pt(text, font, factor):
    return font.getlength(text) / 1000.0 * TABLE_FONT_PT * factor


def unbreakable_pieces(text):
    """Pieces Word keeps on one line: words, with compound words (letter-hyphen-letter) split after the hyphen."""
    pieces = []
    for word in re.split(r"[ \t\n]+", text):
        pieces.extend(re.split(r"(?<=[A-Za-z]-)(?=[A-Za-z])", word))
    return pieces


def column_widths_pt(headers, rows):
    """Each column gets at least its widest unbreakable piece; leftover page width goes to columns that wrap."""
    n_columns = len(headers)
    min_pt = [0.0] * n_columns
    natural_pt = [0.0] * n_columns
    for cells in [headers, *rows]:
        if len(cells) != n_columns:
            raise ValueError(f"Row has {len(cells)} cells, expected {n_columns}: {cells}")
        for column_index, cell in enumerate(cells):
            text, font, factor = shown_cell(cell)
            for piece in unbreakable_pieces(text):
                piece_pt = text_width_pt(piece, font, factor)
                min_pt[column_index] = max(min_pt[column_index], piece_pt)
            cell_pt = text_width_pt(text, font, factor)
            natural_pt[column_index] = max(natural_pt[column_index], cell_pt)
    extra_pt = CELL_PADDING_PT + COLUMN_SLACK_PT
    min_pt = [width + extra_pt for width in min_pt]
    natural_pt = [width + extra_pt for width in natural_pt]
    if sum(min_pt) > TEXT_WIDTH_PT:
        raise ValueError(
            f"Longest words need {sum(min_pt):.0f}pt but the page is {TEXT_WIDTH_PT:.0f}pt wide; "
            f"shorten labels or drop columns. Per-column minimum: {[round(width) for width in min_pt]}"
        )
    if sum(natural_pt) <= TEXT_WIDTH_PT:
        return natural_pt
    spare_pt = TEXT_WIDTH_PT - sum(min_pt)
    stretch_pt = [natural - minimum for natural, minimum in zip(natural_pt, min_pt)]
    total_stretch_pt = sum(stretch_pt)
    widths_pt = []
    for minimum, stretch in zip(min_pt, stretch_pt):
        widths_pt.append(minimum + spare_pt * stretch / total_stretch_pt)
    return widths_pt


def escape_markdown(text):
    """Escape $ and | so Pandoc reads them as text inside a pipe table."""
    escaped = text.replace("$", r"\$")
    escaped = escaped.replace("|", r"\|")
    return escaped


def pipe_table(headers, rows, column_rules):
    """Pandoc pipe table. column_rules set alignment (':---' left, '---:' right); dash counts come from column_widths_pt."""
    if len(column_rules) != len(headers):
        raise ValueError(f"{len(column_rules)} column rules for {len(headers)} columns")
    widths_pt = column_widths_pt(headers, rows)
    rule_cells = []
    for column_rule, width_pt in zip(column_rules, widths_pt):
        if re.fullmatch(r":?-+:?", column_rule) is None:
            raise ValueError(f"Bad column rule {column_rule!r}")
        dashes = "-" * round(width_pt)
        left = ":" if column_rule.startswith(":") else ""
        right = ":" if column_rule.endswith(":") else ""
        rule_cells.append(left + dashes + right)
    lines = ["| " + " | ".join(headers) + " |"]
    lines.append("|" + "|".join(rule_cells) + "|")
    for row in rows:
        lines.append("| " + " | ".join(row) + " |")
    return "\n".join(lines) + "\n"

## Check the shared settings and build the rows

In [6]:
def only_value(values, setting_name):
    """The single value every run shares; raises if the runs disagree."""
    distinct = set(values)
    if len(distinct) != 1:
        raise ValueError(f"Runs disagree on {setting_name}: {sorted(distinct, key=str)}")
    return distinct.pop()


def latex_scientific(value):
    mantissa_text, exponent_text = f"{value:.0e}".split("e")
    exponent = int(exponent_text)
    if float(f"{mantissa_text}e{exponent}") != value:
        raise ValueError(f"{value} needs more than one significant digit")
    if mantissa_text == "1":
        return f"$10^{{{exponent}}}$"
    return f"${mantissa_text} \\times 10^{{{exponent}}}$"


def lora_modules(trainable_shapes):
    modules = set()
    for tensor_name in trainable_shapes:
        if ".lora_A." in tensor_name:
            module_name = tensor_name.split(".lora_A.")[0].split(".")[-1]
            modules.add(module_name)
    return modules


configs = [record["identity"]["config"] for record in run_records.values()]
model_metadata = [record["identity"]["run_metadata"]["model"] for record in run_records.values()]

# Plain respondent-level CE: no beta loss and no KL anchor
if only_value([config["beta_lambda"] for config in configs], "beta_lambda") != 0.0:
    raise ValueError("A run has a beta loss; this table describes CE-only choice training")
if only_value([config["kl_anchor_alpha"] for config in configs], "kl_anchor_alpha") != 0.0:
    raise ValueError("A run has a KL anchor; this table describes CE-only choice training")

lr = only_value([config["lr"] for config in configs], "lr")
num_epochs = only_value([config["num_epochs"] for config in configs], "num_epochs")
lr_schedule = only_value([config["lr_schedule"] for config in configs], "lr_schedule")
weight_decay = only_value([config["weight_decay"] for config in configs], "weight_decay")
grad_clip = only_value([config["grad_clip"] for config in configs], "grad_clip")
seed = only_value([config["seed"] for config in configs], "seed")
batch_sizes = []
for config in configs:
    batch_sizes.extend(config["sample_size_by_dataset"].values())
batch_size = only_value(batch_sizes, "sample_size_by_dataset")
eval_every = only_value([config["eval_every_by_validator"][SELECTION_VALIDATOR] for config in configs], "eval_every")

lora_rank = only_value([metadata["lora_rank"] for metadata in model_metadata], "lora_rank")
weights_dtype = only_value([metadata["weights_dtype"] for metadata in model_metadata], "weights_dtype")
head_temperature = only_value([metadata["head_temperature"] for metadata in model_metadata], "head_temperature")

best_metrics = [record["identity"]["best_metric"] for record in run_records.values()]
best_modes = [record["identity"]["best_mode"] for record in run_records.values()]
if only_value(best_metrics, "best_metric") != f"{SELECTION_VALIDATOR}/beta_mse":
    raise ValueError(f"Checkpoints were not selected on {SELECTION_VALIDATOR}/beta_mse")
if only_value(best_modes, "best_mode") != "min":
    raise ValueError("Checkpoints were not selected by the minimum")

head_dims = []
module_sets = []
for record in run_records.values():
    trainable_shapes = record["identity"]["trainable_shapes"]
    head_dims.append(trainable_shapes[HEAD_WEIGHT_KEY][0])
    module_sets.append(frozenset(lora_modules(trainable_shapes)))
head_dim = only_value(head_dims, "pointer-head dimension")
if only_value(module_sets, "LoRA modules") != EXPECTED_LORA_MODULES:
    raise ValueError(f"LoRA modules differ from EXPECTED_LORA_MODULES: {sorted(module_sets[0])}")

# Base checkpoint revision of each model (shared by its three regimes)
model_revision_text = []
for model_slug in MODELS:
    bases = [run_records[(model_slug, regime_slug)]["identity"]["run_metadata"]["model"]["base"] for regime_slug in TRAINED_SLUGS]
    base = only_value(bases, f"{model_slug} base checkpoint")
    revision = base.split("@")[1]
    model_revision_text.append(f"{MODEL_LABELS[model_slug]} (`{revision[:7]}`)")

schedule_text = (
    f"{LR_SCHEDULE_LABELS[lr_schedule]}: {ONECYCLE_PCT_START:.0%} warmup, {ONECYCLE_ANNEAL} annealing, "
    f"start = peak / {ONECYCLE_DIV_FACTOR:g}, end = start / {latex_scientific(ONECYCLE_FINAL_DIV_FACTOR)}"
)
adamw_text = (
    f"AdamW ($\\beta_1 = {ADAMW_BETAS[0]}$, $\\beta_2 = {ADAMW_BETAS[1]}$, "
    f"$\\epsilon = {latex_scientific(ADAMW_EPS)[1:-1]}$)"
)

rows = [
    ["Base decision models", ", ".join(model_revision_text)],
    ["Weights precision", DTYPE_LABELS[weights_dtype]],
    ["Pointer-head temperature", f"{head_temperature:.1f}"],
    ["Pointer-head dimension", f"{head_dim} (trained in full)"],
    ["LoRA rank $r$", f"{lora_rank}"],
    [r"LoRA scaling $\alpha_{\mathrm{LoRA}}$", f"{LORA_ALPHA_PER_RANK * lora_rank}"],
    ["LoRA dropout", f"{LORA_DROPOUT:g}"],
    ["LoRA target modules", LORA_MODULES_TEXT],
    ["Optimizer", adamw_text],
    ["Peak learning rate", latex_scientific(lr)],
    ["Learning-rate schedule", schedule_text],
    ["Weight decay", f"{weight_decay:g}"],
    ["Gradient-norm clipping", f"{grad_clip:.1f}"],
    ["Epochs", f"{num_epochs}"],
    [r"Mini-batch size $\lvert S \rvert$", f"{batch_size} tasks from a single dataset"],
    ["Random seed", f"{seed}"],
    ["Checkpoint selection", f"Lowest {VALIDATOR_LABEL} $\\mathrm{{MSE}}_\\beta$, evaluated every {eval_every} steps"],
    ["Hardware", HARDWARE_TEXT],
]

# Steps per regime (same for every model) and the selected step of each model, in MODELS order
for regime_slug, regime_label in REGIME_LABELS.items():
    summaries = [run_records[(model_slug, regime_slug)]["summary"] for model_slug in MODELS]
    steps_per_epoch = only_value([summary["steps_per_epoch"] for summary in summaries], f"{regime_slug} steps_per_epoch")
    total_steps = only_value([summary["total_steps"] for summary in summaries], f"{regime_slug} total_steps")
    if total_steps != steps_per_epoch * num_epochs:
        raise ValueError(f"{regime_slug}: total_steps {total_steps} != {steps_per_epoch} x {num_epochs}")
    selected_steps = [f"{summary['best_step']:,}" for summary in summaries]
    rows.append([f"Steps, {regime_label}", f"{steps_per_epoch:,} per epoch ({total_steps:,} total)"])
    rows.append([f"Selected step, {regime_label}", " / ".join(selected_steps)])

table_md = pipe_table(HEADERS, rows, COLUMN_RULES)
display(pd.DataFrame(rows, columns=HEADERS))

,Setting,Value
0,Base decision models,"KEV-0.8B (`bf75a6a`), KEV-4B (`6cfce5c`), KEV-..."
1,Weights precision,bfloat16
2,Pointer-head temperature,1.0
3,Pointer-head dimension,256 (trained in full)
4,LoRA rank $r$,64
5,LoRA scaling $\alpha_{\mathrm{LoRA}}$,128
6,LoRA dropout,0.05
7,LoRA target modules,"All attention, MLP, and Gated DeltaNet project..."
8,Optimizer,"AdamW ($\beta_1 = 0.9$, $\beta_2 = 0.999$, $\e..."
9,Peak learning rate,$5 \times 10^{-6}$


## Save `apptbl-hyperparameters.md` (included by the manuscript)

In [7]:
table_path = ASSET_DIR / f"{ASSET_NAME}.md"
table_path.write_text(table_md, encoding="utf-8")
print(table_md)
print(f"Saved {table_path}")

| Setting | Value |
|:-------------------------------------------------------------------------------------------------------------------------------------------|:-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| Base decision models | KEV-0.8B (`bf75a6a`), KEV-4B (`6cfce5c`), KEV-9B (`db029f0`) |
| Weights precision | bfloat16 |
| Pointer-head temperature | 1.0 |
| Pointer-head dimension | 256 (trained in full) |
| LoRA rank $r$ | 64 |
| LoRA scaling $\alpha_{\mathrm{LoRA}}$ | 128 |
| LoRA dropout | 0.05 |
| LoRA target modules | All attention, MLP, and Gated DeltaNet projections |
| Optimizer | AdamW ($\beta_1 = 0.9$, $\beta_2 = 0.999$, $\epsilon = 10^{-8}$) |
| Peak learning rate | $5 \times 10^{-6}$ |
| 